# 06 — MVP Demo (FastAPI on Colab)

Live preview of Stage 1+2: **upload OPG → YOLOv8 (disease + FDI) → SAM adapter → overview of all lesions → click a tooth to see its segmentation**.

Runs the FastAPI backend on Colab (GPU + Drive checkpoints) and exposes a **public URL** via a Cloudflare quick-tunnel. Runtime: **GPU** (SAM ViT-H).

**Prereq:** `yolov8_dentex.pt`, `yolov8_enum.pt` (or `yolo8_enum.pt`), `sam_vit_h_4b8939.pth`, `adapter_best.pth` in Drive `opg-live/checkpoints/`.

> No OpenRouter key needed for this segment-focused demo. (The GPT-4o click-to-explain endpoint still exists in the backend for later, but the UI now focuses on detection + per-tooth segmentation.)

## Cell 1 — Mount + sync repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/opg-live'
import os
REPO = '/content/opg-live'
if not os.path.exists(REPO):
    !git clone https://github.com/AndreasTopuh/opg-live.git {REPO}
!cd {REPO} && git fetch origin && git reset --hard origin/main && git log --oneline -1

## Cell 2 — Install deps + cloudflared

In [ ]:
%pip install -q fastapi uvicorn python-multipart ultralytics segment-anything pycocotools opencv-python-headless sentence-transformers openai
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared
import torch; print('GPU:', torch.cuda.get_device_name(0))

## Cell 3 — (optional) OpenRouter key for explanations

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ['OPENROUTER_API_KEY'] = userdata.get('OPENROUTER_API_KEY')
    print('OPENROUTER_API_KEY set -> click-to-explain enabled')
except Exception:
    print('No key -> detection+mask works; explanation disabled (set Secret OPENROUTER_API_KEY)')

## Cell 4 — Launch server + public URL
Wait for `PUBLIC URL: https://...trycloudflare.com` then open it. First upload loads the models (~30-60s).

In [ ]:
import subprocess, re, time, os, requests

REPO_DIR = '/home/ec2-user/SageMaker/opg-live'
BACKEND_DIR = f'{REPO_DIR}/backend'
LOG_FILE = '/home/ec2-user/SageMaker/uvicorn.log'

# 1. Download Cloudflare secara lokal (mengatasi Permission Denied)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O {REPO_DIR}/cloudflared
!chmod +x {REPO_DIR}/cloudflared

# 2. Reset Git & Matikan proses lama
!cd {REPO_DIR} && git fetch origin -q && git reset --hard origin/main -q && git log --oneline -1
!pkill -f uvicorn 2>/dev/null; pkill -f cloudflared 2>/dev/null; sleep 2

env = dict(os.environ, DRIVE_ROOT=REPO_DIR)
log = open(LOG_FILE, 'w')

print("⏳ Memulai server FastAPI...")
# 3. Jalankan Uvicorn
subprocess.Popen(['uvicorn', 'main:app', '--host', '0.0.0.0', '--port', '8000'],
                 cwd=BACKEND_DIR, env=env, stdout=log, stderr=subprocess.STDOUT)

up = False
for i in range(20):
    time.sleep(1.5)
    try:
        if requests.get('http://localhost:8000/api/health', timeout=2).ok: 
            up = True
            print('✅ Backend berhasil berjalan!')
            break
    except: pass

if not up:
    print("❌ Gagal memulai server:")
    print(open(LOG_FILE).read()[-3000:])
    raise SystemExit

print("⏳ Membuat URL Publik dengan Cloudflare...")
# 4. Jalankan Cloudflare dari folder lokal
tun = subprocess.Popen([f'{REPO_DIR}/cloudflared', 'tunnel', '--url', 'http://localhost:8000'], 
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

url = None
for line in tun.stdout:
    m = re.search(r'https://[-\w]+\.trycloudflare\.com', line)
    if m: 
        url = m.group(0)
        break

if url:
    print('\n========================================')
    print('⭐ PUBLIC URL ANDA:', url)
    print('Klik tautan di atas untuk membuka aplikasi secara aman.')
    print('========================================\n')
else:
    print("❌ Gagal mendapatkan URL Cloudflare.")

In [ ]:
import subprocess, re, time, os
env = dict(os.environ, DRIVE_ROOT=DRIVE_ROOT)
# start FastAPI (background)
server = subprocess.Popen(['uvicorn', 'main:app', '--host', '0.0.0.0', '--port', '8000'],
                          cwd='/content/opg-live/backend', env=env)
time.sleep(6)
# start cloudflare quick-tunnel (background) and capture the public URL
tun = subprocess.Popen(['cloudflared', 'tunnel', '--url', 'http://localhost:8000'],
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for line in tun.stdout:
    m = re.search(r'https://[-\w]+\.trycloudflare\.com', line)
    if m:
        url = m.group(0); break
print('\n========================================')
print('PUBLIC URL:', url)
print('Open it in your browser, upload an OPG.')
print('========================================')

## ✅ What you get
- **Upload OPG** → **Overview**: all lesions, mask per disease colour + FDI + diagnosis
- **Findings list** → **click a tooth** → its **segmentation** (full-image mask highlight + zoom crop + metrics: conf, mask area, SAM score)
- **▦ Overview (all)** button → back to the full view

This is Stage 1 (detect) → Stage 2 (segment) live, using your own trained checkpoints. Stop the cell to shut down the tunnel.